In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\LoL\AppData\Local\Temp\ipykernel_7160\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
wiki.name

'wikipedia'

In [4]:
from gitdb.fun import chunk_size
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
document = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_db = FAISS.from_documents(document, embeddings)
retriver = vector_db.as_retriever()
retriver


USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019DF13A3710>, search_kwargs={})

In [5]:
from langchain_core.tools import create_retriever_tool
retiriver_tool = create_retriever_tool(retriver,"langsmith_search",
                      "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [6]:
retiriver_tool.name

'langsmith_search'

In [7]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [8]:
tools=[wiki,arxiv,retiriver_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\LoL\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000019DF2799800>, coroutine=<function create_retriever_tool.<loc

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [10]:
from langsmith import Client

# 1. Initialize the official native client
client = Client()

# 2. Pull the prompt directly with the required safety flag
prompt = client.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)

# 3. View the underlying prompt structure
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [11]:
### Agents
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
agent = create_tool_calling_agent(llm, tools, prompt)

## Agent Executer
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMe

In [12]:
agent_executor.invoke({"input":"Tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'What is LangSmith'}`


LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Py

{'input': 'Tell me about langsmith',
 'output': [{'type': 'text',
   'text': '**LangSmith** is an observability, debugging, and evaluation platform built specifically for Large Language Model (LLM) applications and AI agents, developed by **LangChain**.\n\nIt helps developers take LLM-powered applications from prototype to production with confidence.\n\n---\n\n### Key Capabilities\n\n1. **Tracing & Observability**\n   - **Step-by-Step Execution Logs:** Captures nested calls across complex chains, agents, and retrieval pipelines (RAG).\n   - **Deep Visibility:** Logs inputs, outputs, prompt templates, tool calls, token usage, latency, and errors for every single step.\n\n2. **Debugging**\n   - **Inspect Runs & Threads:** View multi-turn agent conversations and granular trace steps to pinpoint unexpected model responses, parsing failures, or hallucinated tool arguments.\n   - **Automated Root-Cause Analysis:** Tools like LangSmith Engine help detect recurring issues in production traces.

In [13]:
agent_executor.invoke({"input":"Tell me about machine learning"})




> Entering new AgentExecutor chain...
[{'type': 'text', 'text': '**Machine Learning (ML)** is a subset of **Artificial Intelligence (AI)** focused on building systems that learn from data and improve their performance over time without being explicitly programmed with rule-based instructions.\n\nInstead of writing code that says *"if X, then do Y,"* developers provide algorithms with data, and the model automatically identifies patterns and rules to make predictions or decisions.\n\n---\n\n### 1. How It Works\nThe basic workflow of a machine learning system involves:\n1. **Data Collection & Preparation:** Gathering, cleaning, and formatting relevant data.\n2. **Feature Engineering:** Selecting or transforming variables that help the algorithm learn.\n3. **Model Selection:** Choosing an appropriate algorithm for the task.\n4. **Training:** Feeding the data into the algorithm to let it adjust its internal parameters (weights/biases) to minimize errors.\n5. **Evaluation:** Testing the m

{'input': 'Tell me about machine learning',
 'output': [{'type': 'text',
   'text': '**Machine Learning (ML)** is a subset of **Artificial Intelligence (AI)** focused on building systems that learn from data and improve their performance over time without being explicitly programmed with rule-based instructions.\n\nInstead of writing code that says *"if X, then do Y,"* developers provide algorithms with data, and the model automatically identifies patterns and rules to make predictions or decisions.\n\n---\n\n### 1. How It Works\nThe basic workflow of a machine learning system involves:\n1. **Data Collection & Preparation:** Gathering, cleaning, and formatting relevant data.\n2. **Feature Engineering:** Selecting or transforming variables that help the algorithm learn.\n3. **Model Selection:** Choosing an appropriate algorithm for the task.\n4. **Training:** Feeding the data into the algorithm to let it adjust its internal parameters (weights/biases) to minimize errors.\n5. **Evaluatio